# AKANE E7：辞書の説明文を段々につなぐ（孫娘→人→哺乳類）

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e6/PLAN_E7.md。

- E6と同じ700個。物の語の説明文（E6で取ったもの）から頭の言葉をたどり、3段まで辞書をつなぎます。
- 途中の語の説明文は、Wiktionaryの全部入りのファイル（ダンプ、約1.5GB）を一度だけ取って読みます（サイトに何度も頼まないので断られません）。
- 各段と「分類に着いたか」を NLI（E6と同じ）で判定します。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
3. 終わると、ドライブの `AKANE_アーキテクチャ/E7_results/E7_results.json` ができます。
4. 時間の見込み：ダンプを取る 5〜15分、読む 20〜30分（この間GPUは使わないので「GPUを使っていません」の警告が出ますが、×で閉じてください）、NLI 5〜10分。途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします。

In [ ]:
import os, json, time, bz2, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E7_results"
os.makedirs(OUT, exist_ok=True)
E6_WK = "/content/drive/MyDrive/AKANE_アーキテクチャ/E6_results/E6_wiktionary.json"
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6と同じ
DUMP_URL = "https://dumps.wikimedia.org/enwiktionary/latest/enwiktionary-latest-pages-articles.xml.bz2"
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)

In [ ]:
# 中身（sims/e6/e7_core.py と同じ）
"""E7の中身（Colabと手元の両方で使う）：説明文の頭の名詞（genus）を取り出し、辞書をたどって網を作る材料をそろえる。"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, DEPTH = 8, 4, 3
PERSON_HEADS = ("one who", "someone who", "somebody who", "a person who", "person who", "one that", "anyone who",
                "a person", "someone", "somebody")
OF_HEADS = {"kind", "type", "sort", "variety", "species", "form", "member", "one", "any", "piece", "item", "genus",
            "class", "breed", "style", "group", "family", "unit", "strain", "subspecies", "category", "instance",
            "example", "specimen", "individual", "portion", "quantity"}
META = {"synonym", "ellipsis", "term", "name", "word", "abbreviation", "plural", "spelling", "who", "something",
        "anything", "that", "which", "this", "one", "any", "form", "use", "sense", "variant", "misspelling", "acronym"}
SKIP_DEF = re.compile(r"^(\(|\[)?(obsolete|archaic|dated|rare|slang|vulgar|offensive|derogatory|historical)\b", re.I)


def genus(defn, nlp):
    """説明文の頭の名詞を1語で返す（取れなければ None）。例: 'A young cat' → 'cat'、'One who wears' → 'person'。"""
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    low = t.lower().lstrip("“\"' ")
    for p in PERSON_HEADS:
        if low.startswith(p + " ") or low == p:
            return "person"
    doc = nlp(t[:200])
    chunks = list(doc.noun_chunks)
    if not chunks:
        return None
    root = chunks[0].root
    lem = root.lemma_.lower()
    if lem in OF_HEADS:  # 'A kind of X' → X、'A member of the cat family' → cat family の頭
        for ch in root.children:
            if ch.dep_ == "prep" and ch.text.lower() == "of":
                objs = [c for c in ch.children if c.dep_ == "pobj"]
                if objs:
                    lem = objs[0].lemma_.lower()
                    break
    if lem == "being" and "human" in chunks[0].text.lower():
        lem = "human"
    if lem == "specie":
        lem = "species"
    if lem in META or not re.fullmatch(r"[a-z][a-z\-]*", lem):
        return None
    return lem


# --- Wiktionaryのダンプ（wikitext）から英語の名詞の意味を取り出す ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), [p for p in parts[1:] if "=" not in p]
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return args[1] if len(args) > 1 else ""
    if name in ARG1_T:
        return args[0] if args else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    return re.sub(r"\s+", " ", t).strip(" ;,.") + "."


def noun_defs_from_wikitext(wt, limit=MAX_DEFS_ENT):
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*Noun\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if not line.startswith("# "):
                continue
            raw = line[2:].strip()
            if SKIP_T.match(raw):
                continue
            d = strip_wikitext(raw)
            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
                out.append(d[:220])
    return out[:limit]


def build_graph(items, defs_of, nlp, log=print):
    """物の語から DEPTH 段まで辞書をたどる。defs_of(word, limit) -> list[str]。
    返り値 G[word] = [{"def": 説明文, "genus": 頭の名詞 or None}]（物の語は8個まで、途中の語は4個まで）。"""
    G = {}
    frontier = sorted(set(it["ent"] for it in items))
    ent_defs = {it["ent"]: it["wk_defs"] for it in items}
    for depth in range(DEPTH + 1):
        nxt = set()
        for w in frontier:
            if w in G:
                continue
            ds = ent_defs.get(w) if (depth == 0 and w in ent_defs) else defs_of(w, MAX_DEFS_MID)
            ds = [d for d in (ds or []) if not SKIP_DEF.match(d)]
            G[w] = [{"def": d, "genus": genus(d, nlp)} for d in ds]
            if depth < DEPTH:
                nxt.update(x["genus"] for x in G[w] if x["genus"] and x["genus"] not in G)
        log(f"段{depth}: 語 {len(frontier)}  次に調べる語 {len(nxt)}")
        frontier = sorted(nxt)
        if not frontier:
            break
    return G


def reach(G, start, depth=DEPTH):
    """start から DEPTH 段までに出てくる語（start含む）。"""
    seen, front = {start}, [start]
    for _ in range(depth):
        nf = []
        for w in front:
            for x in G.get(w, []):
                g = x["genus"]
                if g and g not in seen:
                    seen.add(g); nf.append(g)
        front = nf
    return seen


def nli_jobs(items, G):
    """NLIに聞く組：段（"w" means: 説明文 ⇒ W is a type of genus）と着き（⇒ W is a type of C）。"""
    steps, arrive = set(), set()
    for it in items:
        C = it["catw"]
        for w in reach(G, it["ent"]):
            for k, x in enumerate(G.get(w, [])):
                arrive.add((w, k, C))
                if x["genus"]:
                    steps.add((w, k))
    return sorted(steps), sorted(arrive)


def premise(w, d):
    return f"\"{w}\" means: {d}"


def hyp(w, c):
    return f"{w[:1].upper() + w[1:]} is a type of {c}."


def parse_dump(fobj, log=print, every=500000):
    """Wiktionaryのダンプ（XML）を流し読みし、小文字1語の見出しの英語の名詞の意味を集める。{語: [説明文]}"""
    import xml.etree.ElementTree as ET
    out, n, title, ns = {}, 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and re.fullmatch(r"[a-z][a-z\-]*", title) and el.text and "==English==" in el.text:
                ds = noun_defs_from_wikitext(el.text)
                if ds:
                    out[title] = ds
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(out):,}")
    return out

In [ ]:
ITEMS = json.loads(r'''[{"id": 0, "ent": "barbell", "catw": "tool", "stmt": "Barbell is a type of tool."}, {"id": 1, "ent": "tenement", "catw": "building", "stmt": "Tenement is a type of building."}, {"id": 2, "ent": "porpoise", "catw": "dog", "stmt": "Porpoise is a type of dog."}, {"id": 3, "ent": "juvenile", "catw": "boat", "stmt": "Juvenile is a type of boat."}, {"id": 4, "ent": "barbell", "catw": "mammal", "stmt": "Barbell is a type of mammal."}, {"id": 5, "ent": "grocer", "catw": "living thing", "stmt": "Grocer is a type of living thing."}, {"id": 6, "ent": "gramophone", "catw": "game", "stmt": "Gramophone is a type of game."}, {"id": 7, "ent": "hooch", "catw": "artist", "stmt": "Hooch is a type of artist."}, {"id": 8, "ent": "bilingual", "catw": "person", "stmt": "Bilingual is a type of person."}, {"id": 9, "ent": "minnow", "catw": "disease", "stmt": "Minnow is a type of disease."}, {"id": 10, "ent": "gazelle", "catw": "food", "stmt": "Gazelle is a type of food."}, {"id": 11, "ent": "gasket", "catw": "game", "stmt": "Gasket is a type of game."}, {"id": 12, "ent": "isolationist", "catw": "living thing", "stmt": "Isolationist is a type of living thing."}, {"id": 13, "ent": "clamp", "catw": "tool", "stmt": "Clamp is a type of tool."}, {"id": 14, "ent": "copycat", "catw": "body part", "stmt": "Copycat is a type of body part."}, {"id": 15, "ent": "peshmerga", "catw": "living thing", "stmt": "Peshmerga is a type of living thing."}, {"id": 16, "ent": "tabor", "catw": "boat", "stmt": "Tabor is a type of boat."}, {"id": 17, "ent": "stepmother", "catw": "person", "stmt": "Stepmother is a type of person."}, {"id": 18, "ent": "terrier", "catw": "bird", "stmt": "Terrier is a type of bird."}, {"id": 19, "ent": "preteen", "catw": "metal", "stmt": "Preteen is a type of metal."}, {"id": 20, "ent": "composer", "catw": "person", "stmt": "Composer is a type of person."}, {"id": 21, "ent": "suitor", "catw": "living thing", "stmt": "Suitor is a type of living thing."}, {"id": 22, "ent": "leaker", "catw": "person", "stmt": "Leaker is a type of person."}, {"id": 23, "ent": "actuator", "catw": "machine", "stmt": "Actuator is a type of machine."}, {"id": 24, "ent": "lally", "catw": "tool", "stmt": "Lally is a type of tool."}, {"id": 25, "ent": "epileptic", "catw": "disease", "stmt": "Epileptic is a type of disease."}, {"id": 26, "ent": "pew", "catw": "man-made object", "stmt": "Pew is a type of man-made object."}, {"id": 27, "ent": "microphone", "catw": "equipment", "stmt": "Microphone is a type of equipment."}, {"id": 28, "ent": "felt", "catw": "material", "stmt": "Felt is a type of material."}, {"id": 29, "ent": "primrose", "catw": "living thing", "stmt": "Primrose is a type of living thing."}, {"id": 30, "ent": "quartermaster", "catw": "person", "stmt": "Quartermaster is a type of person."}, {"id": 31, "ent": "delegate", "catw": "person", "stmt": "Delegate is a type of person."}, {"id": 32, "ent": "grubby", "catw": "toy", "stmt": "Grubby is a type of toy."}, {"id": 33, "ent": "prince", "catw": "artist", "stmt": "Prince is a type of artist."}, {"id": 34, "ent": "locomotive", "catw": "man-made object", "stmt": "Locomotive is a type of man-made object."}, {"id": 35, "ent": "lobbyist", "catw": "living thing", "stmt": "Lobbyist is a type of living thing."}, {"id": 36, "ent": "workman", "catw": "footwear", "stmt": "Workman is a type of footwear."}, {"id": 37, "ent": "stupid", "catw": "person", "stmt": "Stupid is a type of person."}, {"id": 38, "ent": "watchman", "catw": "body part", "stmt": "Watchman is a type of body part."}, {"id": 39, "ent": "headliner", "catw": "equipment", "stmt": "Headliner is a type of equipment."}, {"id": 40, "ent": "geranium", "catw": "living thing", "stmt": "Geranium is a type of living thing."}, {"id": 41, "ent": "guerrilla", "catw": "weapon", "stmt": "Guerrilla is a type of weapon."}, {"id": 42, "ent": "sheeting", "catw": "food", "stmt": "Sheeting is a type of food."}, {"id": 43, "ent": "consulate", "catw": "device", "stmt": "Consulate is a type of device."}, {"id": 44, "ent": "dinky", "catw": "artist", "stmt": "Dinky is a type of artist."}, {"id": 45, "ent": "caterer", "catw": "sport", "stmt": "Caterer is a type of sport."}, {"id": 46, "ent": "rickey", "catw": "tree", "stmt": "Rickey is a type of tree."}, {"id": 47, "ent": "headmistress", "catw": "worker", "stmt": "Headmistress is a type of worker."}, {"id": 48, "ent": "educator", "catw": "worker", "stmt": "Educator is a type of worker."}, {"id": 49, "ent": "golf", "catw": "metal", "stmt": "Golf is a type of metal."}, {"id": 50, "ent": "velvet", "catw": "man-made object", "stmt": "Velvet is a type of man-made object."}, {"id": 51, "ent": "biographer", "catw": "disease", "stmt": "Biographer is a type of disease."}, {"id": 52, "ent": "zoologist", "catw": "living thing", "stmt": "Zoologist is a type of living thing."}, {"id": 53, "ent": "doge", "catw": "living thing", "stmt": "Doge is a type of living thing."}, {"id": 54, "ent": "mimic", "catw": "reptile", "stmt": "Mimic is a type of reptile."}, {"id": 55, "ent": "passerby", "catw": "organ", "stmt": "Passerby is a type of organ."}, {"id": 56, "ent": "dieter", "catw": "fish", "stmt": "Dieter is a type of fish."}, {"id": 57, "ent": "niqab", "catw": "reptile", "stmt": "Niqab is a type of reptile."}, {"id": 58, "ent": "bulldozer", "catw": "machine", "stmt": "Bulldozer is a type of machine."}, {"id": 59, "ent": "deflector", "catw": "equipment", "stmt": "Deflector is a type of equipment."}, {"id": 60, "ent": "schoolgirl", "catw": "metal", "stmt": "Schoolgirl is a type of metal."}, {"id": 61, "ent": "falconer", "catw": "sport", "stmt": "Falconer is a type of sport."}, {"id": 62, "ent": "marlin", "catw": "drink", "stmt": "Marlin is a type of drink."}, {"id": 63, "ent": "plumber", "catw": "bird", "stmt": "Plumber is a type of bird."}, {"id": 64, "ent": "sarcophagus", "catw": "man-made object", "stmt": "Sarcophagus is a type of man-made object."}, {"id": 65, "ent": "amputee", "catw": "person", "stmt": "Amputee is a type of person."}, {"id": 66, "ent": "kimono", "catw": "fish", "stmt": "Kimono is a type of fish."}, {"id": 67, "ent": "temp", "catw": "worker", "stmt": "Temp is a type of worker."}, {"id": 68, "ent": "propeller", "catw": "reptile", "stmt": "Propeller is a type of reptile."}, {"id": 69, "ent": "wombat", "catw": "building", "stmt": "Wombat is a type of building."}, {"id": 70, "ent": "paintbrush", "catw": "tool", "stmt": "Paintbrush is a type of tool."}, {"id": 71, "ent": "womanizer", "catw": "body part", "stmt": "Womanizer is a type of body part."}, {"id": 72, "ent": "greenhouse", "catw": "man-made object", "stmt": "Greenhouse is a type of man-made object."}, {"id": 73, "ent": "tactician", "catw": "food", "stmt": "Tactician is a type of food."}, {"id": 74, "ent": "achiever", "catw": "fruit", "stmt": "Achiever is a type of fruit."}, {"id": 75, "ent": "moll", "catw": "person", "stmt": "Moll is a type of person."}, {"id": 76, "ent": "bonsai", "catw": "plant", "stmt": "Bonsai is a type of plant."}, {"id": 77, "ent": "underdog", "catw": "tree", "stmt": "Underdog is a type of tree."}, {"id": 78, "ent": "loom", "catw": "car", "stmt": "Loom is a type of car."}, {"id": 79, "ent": "individualist", "catw": "person", "stmt": "Individualist is a type of person."}, {"id": 80, "ent": "easel", "catw": "sport", "stmt": "Easel is a type of sport."}, {"id": 81, "ent": "finalist", "catw": "athlete", "stmt": "Finalist is a type of athlete."}, {"id": 82, "ent": "sagebrush", "catw": "weapon", "stmt": "Sagebrush is a type of weapon."}, {"id": 83, "ent": "uniform", "catw": "equipment", "stmt": "Uniform is a type of equipment."}, {"id": 84, "ent": "bonsai", "catw": "animal", "stmt": "Bonsai is a type of animal."}, {"id": 85, "ent": "receptionist", "catw": "worker", "stmt": "Receptionist is a type of worker."}, {"id": 86, "ent": "commoner", "catw": "worker", "stmt": "Commoner is a type of worker."}, {"id": 87, "ent": "modernist", "catw": "person", "stmt": "Modernist is a type of person."}, {"id": 88, "ent": "cuppa", "catw": "container", "stmt": "Cuppa is a type of container."}, {"id": 89, "ent": "cornerback", "catw": "athlete", "stmt": "Cornerback is a type of athlete."}, {"id": 90, "ent": "minnow", "catw": "game", "stmt": "Minnow is a type of game."}, {"id": 91, "ent": "neurologist", "catw": "worker", "stmt": "Neurologist is a type of worker."}, {"id": 92, "ent": "kitten", "catw": "mammal", "stmt": "Kitten is a type of mammal."}, {"id": 93, "ent": "undercarriage", "catw": "device", "stmt": "Undercarriage is a type of device."}, {"id": 94, "ent": "reporter", "catw": "worker", "stmt": "Reporter is a type of worker."}, {"id": 95, "ent": "critter", "catw": "equipment", "stmt": "Critter is a type of equipment."}, {"id": 96, "ent": "granddaughter", "catw": "mammal", "stmt": "Granddaughter is a type of mammal."}, {"id": 97, "ent": "crusher", "catw": "machine", "stmt": "Crusher is a type of machine."}, {"id": 98, "ent": "accomplice", "catw": "athlete", "stmt": "Accomplice is a type of athlete."}, {"id": 99, "ent": "dubbing", "catw": "device", "stmt": "Dubbing is a type of device."}, {"id": 100, "ent": "tortoise", "catw": "reptile", "stmt": "Tortoise is a type of reptile."}, {"id": 101, "ent": "jukebox", "catw": "equipment", "stmt": "Jukebox is a type of equipment."}, {"id": 102, "ent": "restaurant", "catw": "building", "stmt": "Restaurant is a type of building."}, {"id": 103, "ent": "sectional", "catw": "man-made object", "stmt": "Sectional is a type of man-made object."}, {"id": 104, "ent": "heifer", "catw": "living thing", "stmt": "Heifer is a type of living thing."}, {"id": 105, "ent": "kea", "catw": "living thing", "stmt": "Kea is a type of living thing."}, {"id": 106, "ent": "minion", "catw": "living thing", "stmt": "Minion is a type of living thing."}, {"id": 107, "ent": "opportunist", "catw": "person", "stmt": "Opportunist is a type of person."}, {"id": 108, "ent": "conformist", "catw": "artist", "stmt": "Conformist is a type of artist."}, {"id": 109, "ent": "burglar", "catw": "body part", "stmt": "Burglar is a type of body part."}, {"id": 110, "ent": "buddy", "catw": "man-made object", "stmt": "Buddy is a type of man-made object."}, {"id": 111, "ent": "wraparound", "catw": "metal", "stmt": "Wraparound is a type of metal."}, {"id": 112, "ent": "vise", "catw": "tool", "stmt": "Vise is a type of tool."}, {"id": 113, "ent": "helicopter", "catw": "man-made object", "stmt": "Helicopter is a type of man-made object."}, {"id": 114, "ent": "calender", "catw": "tool", "stmt": "Calender is a type of tool."}, {"id": 115, "ent": "snob", "catw": "game", "stmt": "Snob is a type of game."}, {"id": 116, "ent": "drunkard", "catw": "living thing", "stmt": "Drunkard is a type of living thing."}, {"id": 117, "ent": "bandicoot", "catw": "disease", "stmt": "Bandicoot is a type of disease."}, {"id": 118, "ent": "musket", "catw": "vehicle", "stmt": "Musket is a type of vehicle."}, {"id": 119, "ent": "grader", "catw": "sport", "stmt": "Grader is a type of sport."}, {"id": 120, "ent": "briefcase", "catw": "vegetable", "stmt": "Briefcase is a type of vegetable."}, {"id": 121, "ent": "alternator", "catw": "vehicle", "stmt": "Alternator is a type of vehicle."}, {"id": 122, "ent": "wallaby", "catw": "plant", "stmt": "Wallaby is a type of plant."}, {"id": 123, "ent": "interceptor", "catw": "aircraft", "stmt": "Interceptor is a type of aircraft."}, {"id": 124, "ent": "carbine", "catw": "equipment", "stmt": "Carbine is a type of equipment."}, {"id": 125, "ent": "spitfire", "catw": "living thing", "stmt": "Spitfire is a type of living thing."}, {"id": 126, "ent": "huntress", "catw": "worker", "stmt": "Huntress is a type of worker."}, {"id": 127, "ent": "heathen", "catw": "flower", "stmt": "Heathen is a type of flower."}, {"id": 128, "ent": "senna", "catw": "man-made object", "stmt": "Senna is a type of man-made object."}, {"id": 129, "ent": "macho", "catw": "animal", "stmt": "Macho is a type of animal."}, {"id": 130, "ent": "beloved", "catw": "insect", "stmt": "Beloved is a type of insect."}, {"id": 131, "ent": "wallaby", "catw": "living thing", "stmt": "Wallaby is a type of living thing."}, {"id": 132, "ent": "accountant", "catw": "game", "stmt": "Accountant is a type of game."}, {"id": 133, "ent": "shogun", "catw": "fabric", "stmt": "Shogun is a type of fabric."}, {"id": 134, "ent": "parsonage", "catw": "boat", "stmt": "Parsonage is a type of boat."}, {"id": 135, "ent": "timepiece", "catw": "tool", "stmt": "Timepiece is a type of tool."}, {"id": 136, "ent": "biter", "catw": "living thing", "stmt": "Biter is a type of living thing."}, {"id": 137, "ent": "rugby", "catw": "equipment", "stmt": "Rugby is a type of equipment."}, {"id": 138, "ent": "bedroom", "catw": "building", "stmt": "Bedroom is a type of building."}, {"id": 139, "ent": "wombat", "catw": "tree", "stmt": "Wombat is a type of tree."}, {"id": 140, "ent": "everlasting", "catw": "game", "stmt": "Everlasting is a type of game."}, {"id": 141, "ent": "auctioneer", "catw": "material", "stmt": "Auctioneer is a type of material."}, {"id": 142, "ent": "theorist", "catw": "person", "stmt": "Theorist is a type of person."}, {"id": 143, "ent": "skater", "catw": "person", "stmt": "Skater is a type of person."}, {"id": 144, "ent": "mariner", "catw": "metal", "stmt": "Mariner is a type of metal."}, {"id": 145, "ent": "freelancer", "catw": "fish", "stmt": "Freelancer is a type of fish."}, {"id": 146, "ent": "dubbing", "catw": "game", "stmt": "Dubbing is a type of game."}, {"id": 147, "ent": "simulator", "catw": "game", "stmt": "Simulator is a type of game."}, {"id": 148, "ent": "circuitry", "catw": "sport", "stmt": "Circuitry is a type of sport."}, {"id": 149, "ent": "galleon", "catw": "boat", "stmt": "Galleon is a type of boat."}, {"id": 150, "ent": "steed", "catw": "animal", "stmt": "Steed is a type of animal."}, {"id": 151, "ent": "mosquito", "catw": "weapon", "stmt": "Mosquito is a type of weapon."}, {"id": 152, "ent": "legate", "catw": "game", "stmt": "Legate is a type of game."}, {"id": 153, "ent": "maxi", "catw": "room", "stmt": "Maxi is a type of room."}, {"id": 154, "ent": "bodybuilder", "catw": "animal", "stmt": "Bodybuilder is a type of animal."}, {"id": 155, "ent": "newlywed", "catw": "living thing", "stmt": "Newlywed is a type of living thing."}, {"id": 156, "ent": "finch", "catw": "material", "stmt": "Finch is a type of material."}, {"id": 157, "ent": "flea", "catw": "vegetable", "stmt": "Flea is a type of vegetable."}, {"id": 158, "ent": "betel", "catw": "tree", "stmt": "Betel is a type of tree."}, {"id": 159, "ent": "microphone", "catw": "sport", "stmt": "Microphone is a type of sport."}, {"id": 160, "ent": "airliner", "catw": "boat", "stmt": "Airliner is a type of boat."}, {"id": 161, "ent": "commodore", "catw": "machine", "stmt": "Commodore is a type of machine."}, {"id": 162, "ent": "billionaire", "catw": "body part", "stmt": "Billionaire is a type of body part."}, {"id": 163, "ent": "roadhouse", "catw": "game", "stmt": "Roadhouse is a type of game."}, {"id": 164, "ent": "hater", "catw": "person", "stmt": "Hater is a type of person."}, {"id": 165, "ent": "marc", "catw": "fruit", "stmt": "Marc is a type of fruit."}, {"id": 166, "ent": "tugboat", "catw": "disease", "stmt": "Tugboat is a type of disease."}, {"id": 167, "ent": "blouse", "catw": "fish", "stmt": "Blouse is a type of fish."}, {"id": 168, "ent": "houseboat", "catw": "device", "stmt": "Houseboat is a type of device."}, {"id": 169, "ent": "bovine", "catw": "toy", "stmt": "Bovine is a type of toy."}, {"id": 170, "ent": "porpoise", "catw": "athlete", "stmt": "Porpoise is a type of athlete."}, {"id": 171, "ent": "everlasting", "catw": "plant", "stmt": "Everlasting is a type of plant."}, {"id": 172, "ent": "tomahawk", "catw": "man-made object", "stmt": "Tomahawk is a type of man-made object."}, {"id": 173, "ent": "national", "catw": "boat", "stmt": "National is a type of boat."}, {"id": 174, "ent": "underpants", "catw": "man-made object", "stmt": "Underpants is a type of man-made object."}, {"id": 175, "ent": "hitchhiker", "catw": "worker", "stmt": "Hitchhiker is a type of worker."}, {"id": 176, "ent": "cutthroat", "catw": "plant", "stmt": "Cutthroat is a type of plant."}, {"id": 177, "ent": "theologian", "catw": "reptile", "stmt": "Theologian is a type of reptile."}, {"id": 178, "ent": "facilitator", "catw": "vegetable", "stmt": "Facilitator is a type of vegetable."}, {"id": 179, "ent": "rickey", "catw": "man-made object", "stmt": "Rickey is a type of man-made object."}, {"id": 180, "ent": "songwriter", "catw": "dish", "stmt": "Songwriter is a type of dish."}, {"id": 181, "ent": "extremist", "catw": "fruit", "stmt": "Extremist is a type of fruit."}, {"id": 182, "ent": "transvestite", "catw": "sport", "stmt": "Transvestite is a type of sport."}, {"id": 183, "ent": "warship", "catw": "fruit", "stmt": "Warship is a type of fruit."}, {"id": 184, "ent": "spokeswoman", "catw": "body part", "stmt": "Spokeswoman is a type of body part."}, {"id": 185, "ent": "brill", "catw": "animal", "stmt": "Brill is a type of animal."}, {"id": 186, "ent": "secessionist", "catw": "metal", "stmt": "Secessionist is a type of metal."}, {"id": 187, "ent": "beloved", "catw": "fabric", "stmt": "Beloved is a type of fabric."}, {"id": 188, "ent": "impala", "catw": "sport", "stmt": "Impala is a type of sport."}, {"id": 189, "ent": "firearm", "catw": "man-made object", "stmt": "Firearm is a type of man-made object."}, {"id": 190, "ent": "mathematician", "catw": "worker", "stmt": "Mathematician is a type of worker."}, {"id": 191, "ent": "buttercup", "catw": "man-made object", "stmt": "Buttercup is a type of man-made object."}, {"id": 192, "ent": "notary", "catw": "game", "stmt": "Notary is a type of game."}, {"id": 193, "ent": "missus", "catw": "boat", "stmt": "Missus is a type of boat."}, {"id": 194, "ent": "freelancer", "catw": "living thing", "stmt": "Freelancer is a type of living thing."}, {"id": 195, "ent": "beret", "catw": "toy", "stmt": "Beret is a type of toy."}, {"id": 196, "ent": "georgette", "catw": "clothing", "stmt": "Georgette is a type of clothing."}, {"id": 197, "ent": "parsonage", "catw": "man-made object", "stmt": "Parsonage is a type of man-made object."}, {"id": 198, "ent": "dongle", "catw": "metal", "stmt": "Dongle is a type of metal."}, {"id": 199, "ent": "spacecraft", "catw": "vehicle", "stmt": "Spacecraft is a type of vehicle."}, {"id": 200, "ent": "eunuch", "catw": "living thing", "stmt": "Eunuch is a type of living thing."}, {"id": 201, "ent": "apparel", "catw": "clothing", "stmt": "Apparel is a type of clothing."}, {"id": 202, "ent": "messenger", "catw": "person", "stmt": "Messenger is a type of person."}, {"id": 203, "ent": "strainer", "catw": "game", "stmt": "Strainer is a type of game."}, {"id": 204, "ent": "bookkeeper", "catw": "living thing", "stmt": "Bookkeeper is a type of living thing."}, {"id": 205, "ent": "clamp", "catw": "musical instrument", "stmt": "Clamp is a type of musical instrument."}, {"id": 206, "ent": "sieve", "catw": "body part", "stmt": "Sieve is a type of body part."}, {"id": 207, "ent": "madame", "catw": "insect", "stmt": "Madame is a type of insect."}, {"id": 208, "ent": "bethel", "catw": "body part", "stmt": "Bethel is a type of body part."}, {"id": 209, "ent": "moray", "catw": "disease", "stmt": "Moray is a type of disease."}, {"id": 210, "ent": "lemur", "catw": "animal", "stmt": "Lemur is a type of animal."}, {"id": 211, "ent": "orangutan", "catw": "fruit", "stmt": "Orangutan is a type of fruit."}, {"id": 212, "ent": "athlete", "catw": "living thing", "stmt": "Athlete is a type of living thing."}, {"id": 213, "ent": "vole", "catw": "worker", "stmt": "Vole is a type of worker."}, {"id": 214, "ent": "peony", "catw": "car", "stmt": "Peony is a type of car."}, {"id": 215, "ent": "undercarriage", "catw": "man-made object", "stmt": "Undercarriage is a type of man-made object."}, {"id": 216, "ent": "blacksmith", "catw": "artist", "stmt": "Blacksmith is a type of artist."}, {"id": 217, "ent": "erica", "catw": "game", "stmt": "Erica is a type of game."}, {"id": 218, "ent": "clinician", "catw": "person", "stmt": "Clinician is a type of person."}, {"id": 219, "ent": "beekeeper", "catw": "person", "stmt": "Beekeeper is a type of person."}, {"id": 220, "ent": "purveyor", "catw": "living thing", "stmt": "Purveyor is a type of living thing."}, {"id": 221, "ent": "carbine", "catw": "man-made object", "stmt": "Carbine is a type of man-made object."}, {"id": 222, "ent": "ani", "catw": "animal", "stmt": "Ani is a type of animal."}, {"id": 223, "ent": "buyer", "catw": "material", "stmt": "Buyer is a type of material."}, {"id": 224, "ent": "bankrupt", "catw": "man-made object", "stmt": "Bankrupt is a type of man-made object."}, {"id": 225, "ent": "landlady", "catw": "flower", "stmt": "Landlady is a type of flower."}, {"id": 226, "ent": "winger", "catw": "aircraft", "stmt": "Winger is a type of aircraft."}, {"id": 227, "ent": "meatball", "catw": "food", "stmt": "Meatball is a type of food."}, {"id": 228, "ent": "autofocus", "catw": "man-made object", "stmt": "Autofocus is a type of man-made object."}, {"id": 229, "ent": "narrator", "catw": "clothing", "stmt": "Narrator is a type of clothing."}, {"id": 230, "ent": "winch", "catw": "musical instrument", "stmt": "Winch is a type of musical instrument."}, {"id": 231, "ent": "hoist", "catw": "car", "stmt": "Hoist is a type of car."}, {"id": 232, "ent": "poodle", "catw": "disease", "stmt": "Poodle is a type of disease."}, {"id": 233, "ent": "hamster", "catw": "athlete", "stmt": "Hamster is a type of athlete."}, {"id": 234, "ent": "finch", "catw": "bird", "stmt": "Finch is a type of bird."}, {"id": 235, "ent": "shrine", "catw": "vehicle", "stmt": "Shrine is a type of vehicle."}, {"id": 236, "ent": "ivy", "catw": "car", "stmt": "Ivy is a type of car."}, {"id": 237, "ent": "pensioner", "catw": "person", "stmt": "Pensioner is a type of person."}, {"id": 238, "ent": "flatmate", "catw": "organ", "stmt": "Flatmate is a type of organ."}, {"id": 239, "ent": "communicator", "catw": "living thing", "stmt": "Communicator is a type of living thing."}, {"id": 240, "ent": "menial", "catw": "plant", "stmt": "Menial is a type of plant."}, {"id": 241, "ent": "grandparent", "catw": "food", "stmt": "Grandparent is a type of food."}, {"id": 242, "ent": "chalet", "catw": "building", "stmt": "Chalet is a type of building."}, {"id": 243, "ent": "avenger", "catw": "vegetable", "stmt": "Avenger is a type of vegetable."}, {"id": 244, "ent": "gazebo", "catw": "fish", "stmt": "Gazebo is a type of fish."}, {"id": 245, "ent": "marten", "catw": "living thing", "stmt": "Marten is a type of living thing."}, {"id": 246, "ent": "uterus", "catw": "container", "stmt": "Uterus is a type of container."}, {"id": 247, "ent": "mockingbird", "catw": "person", "stmt": "Mockingbird is a type of person."}, {"id": 248, "ent": "milkman", "catw": "living thing", "stmt": "Milkman is a type of living thing."}, {"id": 249, "ent": "sauna", "catw": "animal", "stmt": "Sauna is a type of animal."}, {"id": 250, "ent": "aviator", "catw": "man-made object", "stmt": "Aviator is a type of man-made object."}, {"id": 251, "ent": "freeman", "catw": "worker", "stmt": "Freeman is a type of worker."}, {"id": 252, "ent": "salesman", "catw": "fish", "stmt": "Salesman is a type of fish."}, {"id": 253, "ent": "synchrotron", "catw": "device", "stmt": "Synchrotron is a type of device."}, {"id": 254, "ent": "traveler", "catw": "reptile", "stmt": "Traveler is a type of reptile."}, {"id": 255, "ent": "skewer", "catw": "game", "stmt": "Skewer is a type of game."}, {"id": 256, "ent": "sleuth", "catw": "metal", "stmt": "Sleuth is a type of metal."}, {"id": 257, "ent": "abolitionist", "catw": "artist", "stmt": "Abolitionist is a type of artist."}, {"id": 258, "ent": "mantel", "catw": "device", "stmt": "Mantel is a type of device."}, {"id": 259, "ent": "consulate", "catw": "building", "stmt": "Consulate is a type of building."}, {"id": 260, "ent": "minster", "catw": "boat", "stmt": "Minster is a type of boat."}, {"id": 261, "ent": "informer", "catw": "fruit", "stmt": "Informer is a type of fruit."}, {"id": 262, "ent": "waitress", "catw": "food", "stmt": "Waitress is a type of food."}, {"id": 263, "ent": "loudspeaker", "catw": "man-made object", "stmt": "Loudspeaker is a type of man-made object."}, {"id": 264, "ent": "orator", "catw": "footwear", "stmt": "Orator is a type of footwear."}, {"id": 265, "ent": "drummer", "catw": "worker", "stmt": "Drummer is a type of worker."}, {"id": 266, "ent": "dingo", "catw": "animal", "stmt": "Dingo is a type of animal."}, {"id": 267, "ent": "sailboat", "catw": "machine", "stmt": "Sailboat is a type of machine."}, {"id": 268, "ent": "idealist", "catw": "disease", "stmt": "Idealist is a type of disease."}, {"id": 269, "ent": "cactus", "catw": "weapon", "stmt": "Cactus is a type of weapon."}, {"id": 270, "ent": "basset", "catw": "animal", "stmt": "Basset is a type of animal."}, {"id": 271, "ent": "handkerchief", "catw": "man-made object", "stmt": "Handkerchief is a type of man-made object."}, {"id": 272, "ent": "plumber", "catw": "tree", "stmt": "Plumber is a type of tree."}, {"id": 273, "ent": "webcam", "catw": "man-made object", "stmt": "Webcam is a type of man-made object."}, {"id": 274, "ent": "dinghy", "catw": "boat", "stmt": "Dinghy is a type of boat."}, {"id": 275, "ent": "mod", "catw": "metal", "stmt": "Mod is a type of metal."}, {"id": 276, "ent": "erotic", "catw": "flower", "stmt": "Erotic is a type of flower."}, {"id": 277, "ent": "songbird", "catw": "room", "stmt": "Songbird is a type of room."}, {"id": 278, "ent": "sauerkraut", "catw": "fruit", "stmt": "Sauerkraut is a type of fruit."}, {"id": 279, "ent": "plenipotentiary", "catw": "equipment", "stmt": "Plenipotentiary is a type of equipment."}, {"id": 280, "ent": "dispensary", "catw": "man-made object", "stmt": "Dispensary is a type of man-made object."}, {"id": 281, "ent": "fetus", "catw": "man-made object", "stmt": "Fetus is a type of man-made object."}, {"id": 282, "ent": "orchid", "catw": "artist", "stmt": "Orchid is a type of artist."}, {"id": 283, "ent": "basset", "catw": "dog", "stmt": "Basset is a type of dog."}, {"id": 284, "ent": "coordinator", "catw": "person", "stmt": "Coordinator is a type of person."}, {"id": 285, "ent": "taster", "catw": "worker", "stmt": "Taster is a type of worker."}, {"id": 286, "ent": "swot", "catw": "sport", "stmt": "Swot is a type of sport."}, {"id": 287, "ent": "exporter", "catw": "material", "stmt": "Exporter is a type of material."}, {"id": 288, "ent": "underwear", "catw": "clothing", "stmt": "Underwear is a type of clothing."}, {"id": 289, "ent": "pendulum", "catw": "device", "stmt": "Pendulum is a type of device."}, {"id": 290, "ent": "stoner", "catw": "bird", "stmt": "Stoner is a type of bird."}, {"id": 291, "ent": "bartender", "catw": "game", "stmt": "Bartender is a type of game."}, {"id": 292, "ent": "juggler", "catw": "living thing", "stmt": "Juggler is a type of living thing."}, {"id": 293, "ent": "sovereign", "catw": "game", "stmt": "Sovereign is a type of game."}, {"id": 294, "ent": "stepson", "catw": "mammal", "stmt": "Stepson is a type of mammal."}, {"id": 295, "ent": "mongoose", "catw": "living thing", "stmt": "Mongoose is a type of living thing."}, {"id": 296, "ent": "hater", "catw": "dog", "stmt": "Hater is a type of dog."}, {"id": 297, "ent": "cutthroat", "catw": "game", "stmt": "Cutthroat is a type of game."}, {"id": 298, "ent": "pedophile", "catw": "food", "stmt": "Pedophile is a type of food."}, {"id": 299, "ent": "ripper", "catw": "plant", "stmt": "Ripper is a type of plant."}, {"id": 300, "ent": "sultan", "catw": "organ", "stmt": "Sultan is a type of organ."}, {"id": 301, "ent": "assistant", "catw": "dish", "stmt": "Assistant is a type of dish."}, {"id": 302, "ent": "coauthor", "catw": "vegetable", "stmt": "Coauthor is a type of vegetable."}, {"id": 303, "ent": "ladybug", "catw": "equipment", "stmt": "Ladybug is a type of equipment."}, {"id": 304, "ent": "dredge", "catw": "man-made object", "stmt": "Dredge is a type of man-made object."}, {"id": 305, "ent": "clarence", "catw": "vehicle", "stmt": "Clarence is a type of vehicle."}, {"id": 306, "ent": "eft", "catw": "aircraft", "stmt": "Eft is a type of aircraft."}, {"id": 307, "ent": "modernist", "catw": "building", "stmt": "Modernist is a type of building."}, {"id": 308, "ent": "satchel", "catw": "man-made object", "stmt": "Satchel is a type of man-made object."}, {"id": 309, "ent": "invalid", "catw": "game", "stmt": "Invalid is a type of game."}, {"id": 310, "ent": "jogger", "catw": "food", "stmt": "Jogger is a type of food."}, {"id": 311, "ent": "rancher", "catw": "body part", "stmt": "Rancher is a type of body part."}, {"id": 312, "ent": "clary", "catw": "body part", "stmt": "Clary is a type of body part."}, {"id": 313, "ent": "recliner", "catw": "body part", "stmt": "Recliner is a type of body part."}, {"id": 314, "ent": "mercenary", "catw": "worker", "stmt": "Mercenary is a type of worker."}, {"id": 315, "ent": "storeroom", "catw": "vegetable", "stmt": "Storeroom is a type of vegetable."}, {"id": 316, "ent": "protege", "catw": "vegetable", "stmt": "Protege is a type of vegetable."}, {"id": 317, "ent": "handyman", "catw": "weapon", "stmt": "Handyman is a type of weapon."}, {"id": 318, "ent": "wanton", "catw": "living thing", "stmt": "Wanton is a type of living thing."}, {"id": 319, "ent": "bloodhound", "catw": "vegetable", "stmt": "Bloodhound is a type of vegetable."}, {"id": 320, "ent": "crustacean", "catw": "body part", "stmt": "Crustacean is a type of body part."}, {"id": 321, "ent": "electric", "catw": "car", "stmt": "Electric is a type of car."}, {"id": 322, "ent": "itinerant", "catw": "artist", "stmt": "Itinerant is a type of artist."}, {"id": 323, "ent": "outhouse", "catw": "container", "stmt": "Outhouse is a type of container."}, {"id": 324, "ent": "searchlight", "catw": "metal", "stmt": "Searchlight is a type of metal."}, {"id": 325, "ent": "stickler", "catw": "person", "stmt": "Stickler is a type of person."}, {"id": 326, "ent": "snob", "catw": "person", "stmt": "Snob is a type of person."}, {"id": 327, "ent": "erotic", "catw": "living thing", "stmt": "Erotic is a type of living thing."}, {"id": 328, "ent": "violin", "catw": "musical instrument", "stmt": "Violin is a type of musical instrument."}, {"id": 329, "ent": "plover", "catw": "bird", "stmt": "Plover is a type of bird."}, {"id": 330, "ent": "lobbyist", "catw": "boat", "stmt": "Lobbyist is a type of boat."}, {"id": 331, "ent": "motorcyclist", "catw": "footwear", "stmt": "Motorcyclist is a type of footwear."}, {"id": 332, "ent": "harlequin", "catw": "reptile", "stmt": "Harlequin is a type of reptile."}, {"id": 333, "ent": "cheeseburger", "catw": "man-made object", "stmt": "Cheeseburger is a type of man-made object."}, {"id": 334, "ent": "wearer", "catw": "mammal", "stmt": "Wearer is a type of mammal."}, {"id": 335, "ent": "geisha", "catw": "plant", "stmt": "Geisha is a type of plant."}, {"id": 336, "ent": "poppy", "catw": "sport", "stmt": "Poppy is a type of sport."}, {"id": 337, "ent": "bobbin", "catw": "fruit", "stmt": "Bobbin is a type of fruit."}, {"id": 338, "ent": "preacher", "catw": "person", "stmt": "Preacher is a type of person."}, {"id": 339, "ent": "ladybug", "catw": "insect", "stmt": "Ladybug is a type of insect."}, {"id": 340, "ent": "cubby", "catw": "living thing", "stmt": "Cubby is a type of living thing."}, {"id": 341, "ent": "neurotic", "catw": "metal", "stmt": "Neurotic is a type of metal."}, {"id": 342, "ent": "applicator", "catw": "device", "stmt": "Applicator is a type of device."}, {"id": 343, "ent": "hoarder", "catw": "mammal", "stmt": "Hoarder is a type of mammal."}, {"id": 344, "ent": "escapist", "catw": "game", "stmt": "Escapist is a type of game."}, {"id": 345, "ent": "bereaved", "catw": "living thing", "stmt": "Bereaved is a type of living thing."}, {"id": 346, "ent": "termite", "catw": "mammal", "stmt": "Termite is a type of mammal."}, {"id": 347, "ent": "dragonfly", "catw": "fish", "stmt": "Dragonfly is a type of fish."}, {"id": 348, "ent": "razor", "catw": "weapon", "stmt": "Razor is a type of weapon."}, {"id": 349, "ent": "mensch", "catw": "person", "stmt": "Mensch is a type of person."}, {"id": 350, "ent": "kingfisher", "catw": "insect", "stmt": "Kingfisher is a type of insect."}, {"id": 351, "ent": "snob", "catw": "vegetable", "stmt": "Snob is a type of vegetable."}, {"id": 352, "ent": "bock", "catw": "vegetable", "stmt": "Bock is a type of vegetable."}, {"id": 353, "ent": "accelerometer", "catw": "sport", "stmt": "Accelerometer is a type of sport."}, {"id": 354, "ent": "electrode", "catw": "equipment", "stmt": "Electrode is a type of equipment."}, {"id": 355, "ent": "subcontractor", "catw": "food", "stmt": "Subcontractor is a type of food."}, {"id": 356, "ent": "vase", "catw": "container", "stmt": "Vase is a type of container."}, {"id": 357, "ent": "parachute", "catw": "equipment", "stmt": "Parachute is a type of equipment."}, {"id": 358, "ent": "soloist", "catw": "metal", "stmt": "Soloist is a type of metal."}, {"id": 359, "ent": "prankster", "catw": "person", "stmt": "Prankster is a type of person."}, {"id": 360, "ent": "missus", "catw": "person", "stmt": "Missus is a type of person."}, {"id": 361, "ent": "gynecologist", "catw": "living thing", "stmt": "Gynecologist is a type of living thing."}, {"id": 362, "ent": "landowner", "catw": "person", "stmt": "Landowner is a type of person."}, {"id": 363, "ent": "mosquito", "catw": "mammal", "stmt": "Mosquito is a type of mammal."}, {"id": 364, "ent": "decadent", "catw": "dish", "stmt": "Decadent is a type of dish."}, {"id": 365, "ent": "toolbox", "catw": "tool", "stmt": "Toolbox is a type of tool."}, {"id": 366, "ent": "brandy", "catw": "food", "stmt": "Brandy is a type of food."}, {"id": 367, "ent": "housekeeper", "catw": "flower", "stmt": "Housekeeper is a type of flower."}, {"id": 368, "ent": "convener", "catw": "material", "stmt": "Convener is a type of material."}, {"id": 369, "ent": "moa", "catw": "animal", "stmt": "Moa is a type of animal."}, {"id": 370, "ent": "advertiser", "catw": "metal", "stmt": "Advertiser is a type of metal."}, {"id": 371, "ent": "spaniel", "catw": "dog", "stmt": "Spaniel is a type of dog."}, {"id": 372, "ent": "denture", "catw": "man-made object", "stmt": "Denture is a type of man-made object."}, {"id": 373, "ent": "pigeon", "catw": "person", "stmt": "Pigeon is a type of person."}, {"id": 374, "ent": "chrysalis", "catw": "body part", "stmt": "Chrysalis is a type of body part."}, {"id": 375, "ent": "vizier", "catw": "athlete", "stmt": "Vizier is a type of athlete."}, {"id": 376, "ent": "bandit", "catw": "device", "stmt": "Bandit is a type of device."}, {"id": 377, "ent": "dominatrix", "catw": "living thing", "stmt": "Dominatrix is a type of living thing."}, {"id": 378, "ent": "protectionist", "catw": "plant", "stmt": "Protectionist is a type of plant."}, {"id": 379, "ent": "cyborg", "catw": "body part", "stmt": "Cyborg is a type of body part."}, {"id": 380, "ent": "manger", "catw": "worker", "stmt": "Manger is a type of worker."}, {"id": 381, "ent": "stowaway", "catw": "vegetable", "stmt": "Stowaway is a type of vegetable."}, {"id": 382, "ent": "stingray", "catw": "living thing", "stmt": "Stingray is a type of living thing."}, {"id": 383, "ent": "immigrant", "catw": "game", "stmt": "Immigrant is a type of game."}, {"id": 384, "ent": "peripheral", "catw": "animal", "stmt": "Peripheral is a type of animal."}, {"id": 385, "ent": "corgi", "catw": "clothing", "stmt": "Corgi is a type of clothing."}, {"id": 386, "ent": "trusty", "catw": "game", "stmt": "Trusty is a type of game."}, {"id": 387, "ent": "inventor", "catw": "drink", "stmt": "Inventor is a type of drink."}, {"id": 388, "ent": "narc", "catw": "person", "stmt": "Narc is a type of person."}, {"id": 389, "ent": "brazier", "catw": "drink", "stmt": "Brazier is a type of drink."}, {"id": 390, "ent": "curate", "catw": "living thing", "stmt": "Curate is a type of living thing."}, {"id": 391, "ent": "stagecoach", "catw": "vehicle", "stmt": "Stagecoach is a type of vehicle."}, {"id": 392, "ent": "molester", "catw": "mammal", "stmt": "Molester is a type of mammal."}, {"id": 393, "ent": "paramedic", "catw": "worker", "stmt": "Paramedic is a type of worker."}, {"id": 394, "ent": "inventor", "catw": "body part", "stmt": "Inventor is a type of body part."}, {"id": 395, "ent": "synagogue", "catw": "vegetable", "stmt": "Synagogue is a type of vegetable."}, {"id": 396, "ent": "adolescent", "catw": "mammal", "stmt": "Adolescent is a type of mammal."}, {"id": 397, "ent": "contrarian", "catw": "metal", "stmt": "Contrarian is a type of metal."}, {"id": 398, "ent": "swimsuit", "catw": "equipment", "stmt": "Swimsuit is a type of equipment."}, {"id": 399, "ent": "shrine", "catw": "container", "stmt": "Shrine is a type of container."}, {"id": 400, "ent": "manatee", "catw": "vehicle", "stmt": "Manatee is a type of vehicle."}, {"id": 401, "ent": "toddler", "catw": "tool", "stmt": "Toddler is a type of tool."}, {"id": 402, "ent": "introvert", "catw": "worker", "stmt": "Introvert is a type of worker."}, {"id": 403, "ent": "copywriter", "catw": "clothing", "stmt": "Copywriter is a type of clothing."}, {"id": 404, "ent": "vigilante", "catw": "person", "stmt": "Vigilante is a type of person."}, {"id": 405, "ent": "cuttlefish", "catw": "animal", "stmt": "Cuttlefish is a type of animal."}, {"id": 406, "ent": "alderman", "catw": "athlete", "stmt": "Alderman is a type of athlete."}, {"id": 407, "ent": "hater", "catw": "boat", "stmt": "Hater is a type of boat."}, {"id": 408, "ent": "airship", "catw": "dog", "stmt": "Airship is a type of dog."}, {"id": 409, "ent": "benefactor", "catw": "person", "stmt": "Benefactor is a type of person."}, {"id": 410, "ent": "financier", "catw": "worker", "stmt": "Financier is a type of worker."}, {"id": 411, "ent": "trader", "catw": "worker", "stmt": "Trader is a type of worker."}, {"id": 412, "ent": "anti", "catw": "living thing", "stmt": "Anti is a type of living thing."}, {"id": 413, "ent": "sarcophagus", "catw": "tool", "stmt": "Sarcophagus is a type of tool."}, {"id": 414, "ent": "paleontologist", "catw": "worker", "stmt": "Paleontologist is a type of worker."}, {"id": 415, "ent": "accordion", "catw": "fish", "stmt": "Accordion is a type of fish."}, {"id": 416, "ent": "sphincter", "catw": "disease", "stmt": "Sphincter is a type of disease."}, {"id": 417, "ent": "detainee", "catw": "living thing", "stmt": "Detainee is a type of living thing."}, {"id": 418, "ent": "pleaser", "catw": "game", "stmt": "Pleaser is a type of game."}, {"id": 419, "ent": "princess", "catw": "person", "stmt": "Princess is a type of person."}, {"id": 420, "ent": "paramedic", "catw": "dish", "stmt": "Paramedic is a type of dish."}, {"id": 421, "ent": "rink", "catw": "sport", "stmt": "Rink is a type of sport."}, {"id": 422, "ent": "propagandist", "catw": "person", "stmt": "Propagandist is a type of person."}, {"id": 423, "ent": "confidante", "catw": "person", "stmt": "Confidante is a type of person."}, {"id": 424, "ent": "stockbroker", "catw": "footwear", "stmt": "Stockbroker is a type of footwear."}, {"id": 425, "ent": "drosophila", "catw": "food", "stmt": "Drosophila is a type of food."}, {"id": 426, "ent": "pacifist", "catw": "insect", "stmt": "Pacifist is a type of insect."}, {"id": 427, "ent": "suitor", "catw": "game", "stmt": "Suitor is a type of game."}, {"id": 428, "ent": "apologist", "catw": "worker", "stmt": "Apologist is a type of worker."}, {"id": 429, "ent": "clubhouse", "catw": "room", "stmt": "Clubhouse is a type of room."}, {"id": 430, "ent": "strategist", "catw": "person", "stmt": "Strategist is a type of person."}, {"id": 431, "ent": "jeep", "catw": "container", "stmt": "Jeep is a type of container."}, {"id": 432, "ent": "survivalist", "catw": "fish", "stmt": "Survivalist is a type of fish."}, {"id": 433, "ent": "vermouth", "catw": "plant", "stmt": "Vermouth is a type of plant."}, {"id": 434, "ent": "contrarian", "catw": "animal", "stmt": "Contrarian is a type of animal."}, {"id": 435, "ent": "cattle", "catw": "animal", "stmt": "Cattle is a type of animal."}, {"id": 436, "ent": "pensioner", "catw": "sport", "stmt": "Pensioner is a type of sport."}, {"id": 437, "ent": "marlin", "catw": "living thing", "stmt": "Marlin is a type of living thing."}, {"id": 438, "ent": "mockingbird", "catw": "bird", "stmt": "Mockingbird is a type of bird."}, {"id": 439, "ent": "dandelion", "catw": "mammal", "stmt": "Dandelion is a type of mammal."}, {"id": 440, "ent": "meatball", "catw": "game", "stmt": "Meatball is a type of game."}, {"id": 441, "ent": "chime", "catw": "organ", "stmt": "Chime is a type of organ."}, {"id": 442, "ent": "hornet", "catw": "worker", "stmt": "Hornet is a type of worker."}, {"id": 443, "ent": "wildflower", "catw": "living thing", "stmt": "Wildflower is a type of living thing."}, {"id": 444, "ent": "gyroscope", "catw": "machine", "stmt": "Gyroscope is a type of machine."}, {"id": 445, "ent": "lender", "catw": "bird", "stmt": "Lender is a type of bird."}, {"id": 446, "ent": "kangaroo", "catw": "athlete", "stmt": "Kangaroo is a type of athlete."}, {"id": 447, "ent": "calico", "catw": "fabric", "stmt": "Calico is a type of fabric."}, {"id": 448, "ent": "tongs", "catw": "tool", "stmt": "Tongs is a type of tool."}, {"id": 449, "ent": "cadet", "catw": "dish", "stmt": "Cadet is a type of dish."}, {"id": 450, "ent": "repairman", "catw": "fruit", "stmt": "Repairman is a type of fruit."}, {"id": 451, "ent": "clubhouse", "catw": "machine", "stmt": "Clubhouse is a type of machine."}, {"id": 452, "ent": "hummingbird", "catw": "animal", "stmt": "Hummingbird is a type of animal."}, {"id": 453, "ent": "bison", "catw": "mammal", "stmt": "Bison is a type of mammal."}, {"id": 454, "ent": "piglet", "catw": "mammal", "stmt": "Piglet is a type of mammal."}, {"id": 455, "ent": "hyena", "catw": "plant", "stmt": "Hyena is a type of plant."}, {"id": 456, "ent": "wife", "catw": "mammal", "stmt": "Wife is a type of mammal."}, {"id": 457, "ent": "berk", "catw": "living thing", "stmt": "Berk is a type of living thing."}, {"id": 458, "ent": "transistor", "catw": "fish", "stmt": "Transistor is a type of fish."}, {"id": 459, "ent": "firstborn", "catw": "sport", "stmt": "Firstborn is a type of sport."}, {"id": 460, "ent": "interviewee", "catw": "worker", "stmt": "Interviewee is a type of worker."}, {"id": 461, "ent": "surrealist", "catw": "fish", "stmt": "Surrealist is a type of fish."}, {"id": 462, "ent": "corrective", "catw": "man-made object", "stmt": "Corrective is a type of man-made object."}, {"id": 463, "ent": "searchlight", "catw": "equipment", "stmt": "Searchlight is a type of equipment."}, {"id": 464, "ent": "bookkeeper", "catw": "reptile", "stmt": "Bookkeeper is a type of reptile."}, {"id": 465, "ent": "takin", "catw": "fish", "stmt": "Takin is a type of fish."}, {"id": 466, "ent": "therapist", "catw": "living thing", "stmt": "Therapist is a type of living thing."}, {"id": 467, "ent": "communicator", "catw": "person", "stmt": "Communicator is a type of person."}, {"id": 468, "ent": "bereaved", "catw": "game", "stmt": "Bereaved is a type of game."}, {"id": 469, "ent": "apparel", "catw": "weapon", "stmt": "Apparel is a type of weapon."}, {"id": 470, "ent": "boardroom", "catw": "room", "stmt": "Boardroom is a type of room."}, {"id": 471, "ent": "goldilocks", "catw": "flower", "stmt": "Goldilocks is a type of flower."}, {"id": 472, "ent": "manatee", "catw": "food", "stmt": "Manatee is a type of food."}, {"id": 473, "ent": "aloe", "catw": "room", "stmt": "Aloe is a type of room."}, {"id": 474, "ent": "congressman", "catw": "flower", "stmt": "Congressman is a type of flower."}, {"id": 475, "ent": "ecologist", "catw": "worker", "stmt": "Ecologist is a type of worker."}, {"id": 476, "ent": "dahlia", "catw": "plant", "stmt": "Dahlia is a type of plant."}, {"id": 477, "ent": "lodger", "catw": "plant", "stmt": "Lodger is a type of plant."}, {"id": 478, "ent": "flea", "catw": "animal", "stmt": "Flea is a type of animal."}, {"id": 479, "ent": "hillbilly", "catw": "living thing", "stmt": "Hillbilly is a type of living thing."}, {"id": 480, "ent": "stewardess", "catw": "equipment", "stmt": "Stewardess is a type of equipment."}, {"id": 481, "ent": "helmsman", "catw": "athlete", "stmt": "Helmsman is a type of athlete."}, {"id": 482, "ent": "jasmine", "catw": "vegetable", "stmt": "Jasmine is a type of vegetable."}, {"id": 483, "ent": "palmetto", "catw": "musical instrument", "stmt": "Palmetto is a type of musical instrument."}, {"id": 484, "ent": "rapporteur", "catw": "organ", "stmt": "Rapporteur is a type of organ."}, {"id": 485, "ent": "bobcat", "catw": "mammal", "stmt": "Bobcat is a type of mammal."}, {"id": 486, "ent": "cutler", "catw": "living thing", "stmt": "Cutler is a type of living thing."}, {"id": 487, "ent": "hearse", "catw": "man-made object", "stmt": "Hearse is a type of man-made object."}, {"id": 488, "ent": "spastic", "catw": "reptile", "stmt": "Spastic is a type of reptile."}, {"id": 489, "ent": "alcoholic", "catw": "drink", "stmt": "Alcoholic is a type of drink."}, {"id": 490, "ent": "longhorn", "catw": "tree", "stmt": "Longhorn is a type of tree."}, {"id": 491, "ent": "beagle", "catw": "sport", "stmt": "Beagle is a type of sport."}, {"id": 492, "ent": "impressionist", "catw": "fabric", "stmt": "Impressionist is a type of fabric."}, {"id": 493, "ent": "inpatient", "catw": "fruit", "stmt": "Inpatient is a type of fruit."}, {"id": 494, "ent": "emir", "catw": "boat", "stmt": "Emir is a type of boat."}, {"id": 495, "ent": "snapdragon", "catw": "living thing", "stmt": "Snapdragon is a type of living thing."}, {"id": 496, "ent": "stockholder", "catw": "furniture", "stmt": "Stockholder is a type of furniture."}, {"id": 497, "ent": "schizophrenic", "catw": "sport", "stmt": "Schizophrenic is a type of sport."}, {"id": 498, "ent": "rapporteur", "catw": "person", "stmt": "Rapporteur is a type of person."}, {"id": 499, "ent": "ombudsman", "catw": "living thing", "stmt": "Ombudsman is a type of living thing."}, {"id": 500, "ent": "cleric", "catw": "plant", "stmt": "Cleric is a type of plant."}, {"id": 501, "ent": "patriot", "catw": "aircraft", "stmt": "Patriot is a type of aircraft."}, {"id": 502, "ent": "homophobe", "catw": "sport", "stmt": "Homophobe is a type of sport."}, {"id": 503, "ent": "rifle", "catw": "animal", "stmt": "Rifle is a type of animal."}, {"id": 504, "ent": "phytoplankton", "catw": "plant", "stmt": "Phytoplankton is a type of plant."}, {"id": 505, "ent": "caveman", "catw": "person", "stmt": "Caveman is a type of person."}, {"id": 506, "ent": "exporter", "catw": "drink", "stmt": "Exporter is a type of drink."}, {"id": 507, "ent": "sibling", "catw": "mammal", "stmt": "Sibling is a type of mammal."}, {"id": 508, "ent": "limousine", "catw": "clothing", "stmt": "Limousine is a type of clothing."}, {"id": 509, "ent": "lyre", "catw": "man-made object", "stmt": "Lyre is a type of man-made object."}, {"id": 510, "ent": "audiovisual", "catw": "material", "stmt": "Audiovisual is a type of material."}, {"id": 511, "ent": "communicator", "catw": "sport", "stmt": "Communicator is a type of sport."}, {"id": 512, "ent": "bathrobe", "catw": "metal", "stmt": "Bathrobe is a type of metal."}, {"id": 513, "ent": "thwart", "catw": "device", "stmt": "Thwart is a type of device."}, {"id": 514, "ent": "stork", "catw": "aircraft", "stmt": "Stork is a type of aircraft."}, {"id": 515, "ent": "marksman", "catw": "game", "stmt": "Marksman is a type of game."}, {"id": 516, "ent": "confessional", "catw": "toy", "stmt": "Confessional is a type of toy."}, {"id": 517, "ent": "monastery", "catw": "container", "stmt": "Monastery is a type of container."}, {"id": 518, "ent": "headdress", "catw": "flower", "stmt": "Headdress is a type of flower."}, {"id": 519, "ent": "statesman", "catw": "living thing", "stmt": "Statesman is a type of living thing."}, {"id": 520, "ent": "milkman", "catw": "vegetable", "stmt": "Milkman is a type of vegetable."}, {"id": 521, "ent": "thermometer", "catw": "tool", "stmt": "Thermometer is a type of tool."}, {"id": 522, "ent": "expressionist", "catw": "game", "stmt": "Expressionist is a type of game."}, {"id": 523, "ent": "perpetrator", "catw": "insect", "stmt": "Perpetrator is a type of insect."}, {"id": 524, "ent": "coupe", "catw": "container", "stmt": "Coupe is a type of container."}, {"id": 525, "ent": "councilman", "catw": "worker", "stmt": "Councilman is a type of worker."}, {"id": 526, "ent": "authoritarian", "catw": "sport", "stmt": "Authoritarian is a type of sport."}, {"id": 527, "ent": "satirist", "catw": "artist", "stmt": "Satirist is a type of artist."}, {"id": 528, "ent": "antelope", "catw": "living thing", "stmt": "Antelope is a type of living thing."}, {"id": 529, "ent": "scraper", "catw": "device", "stmt": "Scraper is a type of device."}, {"id": 530, "ent": "ballroom", "catw": "sport", "stmt": "Ballroom is a type of sport."}, {"id": 531, "ent": "dunce", "catw": "flower", "stmt": "Dunce is a type of flower."}, {"id": 532, "ent": "hiker", "catw": "athlete", "stmt": "Hiker is a type of athlete."}, {"id": 533, "ent": "playmate", "catw": "mammal", "stmt": "Playmate is a type of mammal."}, {"id": 534, "ent": "agitator", "catw": "vegetable", "stmt": "Agitator is a type of vegetable."}, {"id": 535, "ent": "stiletto", "catw": "footwear", "stmt": "Stiletto is a type of footwear."}, {"id": 536, "ent": "clock", "catw": "machine", "stmt": "Clock is a type of machine."}, {"id": 537, "ent": "bankrupt", "catw": "person", "stmt": "Bankrupt is a type of person."}, {"id": 538, "ent": "sophomore", "catw": "boat", "stmt": "Sophomore is a type of boat."}, {"id": 539, "ent": "hillbilly", "catw": "material", "stmt": "Hillbilly is a type of material."}, {"id": 540, "ent": "yarrow", "catw": "flower", "stmt": "Yarrow is a type of flower."}, {"id": 541, "ent": "caveman", "catw": "living thing", "stmt": "Caveman is a type of living thing."}, {"id": 542, "ent": "battleship", "catw": "person", "stmt": "Battleship is a type of person."}, {"id": 543, "ent": "ratchet", "catw": "tool", "stmt": "Ratchet is a type of tool."}, {"id": 544, "ent": "rival", "catw": "weapon", "stmt": "Rival is a type of weapon."}, {"id": 545, "ent": "candlestick", "catw": "device", "stmt": "Candlestick is a type of device."}, {"id": 546, "ent": "helmsman", "catw": "boat", "stmt": "Helmsman is a type of boat."}, {"id": 547, "ent": "suitor", "catw": "person", "stmt": "Suitor is a type of person."}, {"id": 548, "ent": "greenhouse", "catw": "device", "stmt": "Greenhouse is a type of device."}, {"id": 549, "ent": "socialist", "catw": "person", "stmt": "Socialist is a type of person."}, {"id": 550, "ent": "prince", "catw": "person", "stmt": "Prince is a type of person."}, {"id": 551, "ent": "pulley", "catw": "footwear", "stmt": "Pulley is a type of footwear."}, {"id": 552, "ent": "parachute", "catw": "device", "stmt": "Parachute is a type of device."}, {"id": 553, "ent": "crocodile", "catw": "body part", "stmt": "Crocodile is a type of body part."}, {"id": 554, "ent": "innocent", "catw": "man-made object", "stmt": "Innocent is a type of man-made object."}, {"id": 555, "ent": "conch", "catw": "living thing", "stmt": "Conch is a type of living thing."}, {"id": 556, "ent": "electrode", "catw": "car", "stmt": "Electrode is a type of car."}, {"id": 557, "ent": "raglan", "catw": "furniture", "stmt": "Raglan is a type of furniture."}, {"id": 558, "ent": "nacho", "catw": "material", "stmt": "Nacho is a type of material."}, {"id": 559, "ent": "surgeon", "catw": "worker", "stmt": "Surgeon is a type of worker."}, {"id": 560, "ent": "artist", "catw": "fish", "stmt": "Artist is a type of fish."}, {"id": 561, "ent": "claimant", "catw": "disease", "stmt": "Claimant is a type of disease."}, {"id": 562, "ent": "ibis", "catw": "footwear", "stmt": "Ibis is a type of footwear."}, {"id": 563, "ent": "notary", "catw": "person", "stmt": "Notary is a type of person."}, {"id": 564, "ent": "motorboat", "catw": "machine", "stmt": "Motorboat is a type of machine."}, {"id": 565, "ent": "loom", "catw": "man-made object", "stmt": "Loom is a type of man-made object."}, {"id": 566, "ent": "firearm", "catw": "weapon", "stmt": "Firearm is a type of weapon."}, {"id": 567, "ent": "haggis", "catw": "organ", "stmt": "Haggis is a type of organ."}, {"id": 568, "ent": "brogan", "catw": "food", "stmt": "Brogan is a type of food."}, {"id": 569, "ent": "tam", "catw": "dish", "stmt": "Tam is a type of dish."}, {"id": 570, "ent": "motorcyclist", "catw": "sport", "stmt": "Motorcyclist is a type of sport."}, {"id": 571, "ent": "alumnus", "catw": "plant", "stmt": "Alumnus is a type of plant."}, {"id": 572, "ent": "dildo", "catw": "equipment", "stmt": "Dildo is a type of equipment."}, {"id": 573, "ent": "tights", "catw": "vegetable", "stmt": "Tights is a type of vegetable."}, {"id": 574, "ent": "doorbell", "catw": "boat", "stmt": "Doorbell is a type of boat."}, {"id": 575, "ent": "letterman", "catw": "reptile", "stmt": "Letterman is a type of reptile."}, {"id": 576, "ent": "athlete", "catw": "worker", "stmt": "Athlete is a type of worker."}, {"id": 577, "ent": "hostage", "catw": "game", "stmt": "Hostage is a type of game."}, {"id": 578, "ent": "empty", "catw": "container", "stmt": "Empty is a type of container."}, {"id": 579, "ent": "spacecraft", "catw": "machine", "stmt": "Spacecraft is a type of machine."}, {"id": 580, "ent": "kilt", "catw": "boat", "stmt": "Kilt is a type of boat."}, {"id": 581, "ent": "shook", "catw": "body part", "stmt": "Shook is a type of body part."}, {"id": 582, "ent": "headscarf", "catw": "clothing", "stmt": "Headscarf is a type of clothing."}, {"id": 583, "ent": "geranium", "catw": "animal", "stmt": "Geranium is a type of animal."}, {"id": 584, "ent": "mangrove", "catw": "weapon", "stmt": "Mangrove is a type of weapon."}, {"id": 585, "ent": "librarian", "catw": "food", "stmt": "Librarian is a type of food."}, {"id": 586, "ent": "restaurateur", "catw": "athlete", "stmt": "Restaurateur is a type of athlete."}, {"id": 587, "ent": "perfectionist", "catw": "living thing", "stmt": "Perfectionist is a type of living thing."}, {"id": 588, "ent": "flashlight", "catw": "tool", "stmt": "Flashlight is a type of tool."}, {"id": 589, "ent": "groundhog", "catw": "man-made object", "stmt": "Groundhog is a type of man-made object."}, {"id": 590, "ent": "contemplative", "catw": "footwear", "stmt": "Contemplative is a type of footwear."}, {"id": 591, "ent": "robber", "catw": "living thing", "stmt": "Robber is a type of living thing."}, {"id": 592, "ent": "tequila", "catw": "plant", "stmt": "Tequila is a type of plant."}, {"id": 593, "ent": "expert", "catw": "person", "stmt": "Expert is a type of person."}, {"id": 594, "ent": "wheelchair", "catw": "game", "stmt": "Wheelchair is a type of game."}, {"id": 595, "ent": "cuttlefish", "catw": "artist", "stmt": "Cuttlefish is a type of artist."}, {"id": 596, "ent": "traveler", "catw": "game", "stmt": "Traveler is a type of game."}, {"id": 597, "ent": "thermos", "catw": "plant", "stmt": "Thermos is a type of plant."}, {"id": 598, "ent": "customer", "catw": "weapon", "stmt": "Customer is a type of weapon."}, {"id": 599, "ent": "loner", "catw": "living thing", "stmt": "Loner is a type of living thing."}, {"id": 600, "ent": "slipknot", "catw": "material", "stmt": "Slipknot is a type of material."}, {"id": 601, "ent": "landlady", "catw": "game", "stmt": "Landlady is a type of game."}, {"id": 602, "ent": "oppressor", "catw": "device", "stmt": "Oppressor is a type of device."}, {"id": 603, "ent": "alderman", "catw": "boat", "stmt": "Alderman is a type of boat."}, {"id": 604, "ent": "fife", "catw": "fabric", "stmt": "Fife is a type of fabric."}, {"id": 605, "ent": "loudspeaker", "catw": "disease", "stmt": "Loudspeaker is a type of disease."}, {"id": 606, "ent": "revisionist", "catw": "artist", "stmt": "Revisionist is a type of artist."}, {"id": 607, "ent": "plenipotentiary", "catw": "mammal", "stmt": "Plenipotentiary is a type of mammal."}, {"id": 608, "ent": "houseboat", "catw": "vehicle", "stmt": "Houseboat is a type of vehicle."}, {"id": 609, "ent": "optimist", "catw": "body part", "stmt": "Optimist is a type of body part."}, {"id": 610, "ent": "primrose", "catw": "flower", "stmt": "Primrose is a type of flower."}, {"id": 611, "ent": "moray", "catw": "insect", "stmt": "Moray is a type of insect."}, {"id": 612, "ent": "roebuck", "catw": "animal", "stmt": "Roebuck is a type of animal."}, {"id": 613, "ent": "cafe", "catw": "man-made object", "stmt": "Cafe is a type of man-made object."}, {"id": 614, "ent": "proletarian", "catw": "worker", "stmt": "Proletarian is a type of worker."}, {"id": 615, "ent": "limiter", "catw": "man-made object", "stmt": "Limiter is a type of man-made object."}, {"id": 616, "ent": "brogan", "catw": "man-made object", "stmt": "Brogan is a type of man-made object."}, {"id": 617, "ent": "wingman", "catw": "game", "stmt": "Wingman is a type of game."}, {"id": 618, "ent": "robber", "catw": "game", "stmt": "Robber is a type of game."}, {"id": 619, "ent": "aquarium", "catw": "boat", "stmt": "Aquarium is a type of boat."}, {"id": 620, "ent": "handkerchief", "catw": "device", "stmt": "Handkerchief is a type of device."}, {"id": 621, "ent": "legislator", "catw": "worker", "stmt": "Legislator is a type of worker."}, {"id": 622, "ent": "preteen", "catw": "person", "stmt": "Preteen is a type of person."}, {"id": 623, "ent": "individualist", "catw": "worker", "stmt": "Individualist is a type of worker."}, {"id": 624, "ent": "detonator", "catw": "furniture", "stmt": "Detonator is a type of furniture."}, {"id": 625, "ent": "banjo", "catw": "fish", "stmt": "Banjo is a type of fish."}, {"id": 626, "ent": "neurosurgeon", "catw": "footwear", "stmt": "Neurosurgeon is a type of footwear."}, {"id": 627, "ent": "sunflower", "catw": "living thing", "stmt": "Sunflower is a type of living thing."}, {"id": 628, "ent": "kylie", "catw": "aircraft", "stmt": "Kylie is a type of aircraft."}, {"id": 629, "ent": "nawab", "catw": "clothing", "stmt": "Nawab is a type of clothing."}, {"id": 630, "ent": "assistant", "catw": "person", "stmt": "Assistant is a type of person."}, {"id": 631, "ent": "cantilever", "catw": "body part", "stmt": "Cantilever is a type of body part."}, {"id": 632, "ent": "libertine", "catw": "fish", "stmt": "Libertine is a type of fish."}, {"id": 633, "ent": "chronicler", "catw": "artist", "stmt": "Chronicler is a type of artist."}, {"id": 634, "ent": "earl", "catw": "boat", "stmt": "Earl is a type of boat."}, {"id": 635, "ent": "abbess", "catw": "building", "stmt": "Abbess is a type of building."}, {"id": 636, "ent": "murine", "catw": "food", "stmt": "Murine is a type of food."}, {"id": 637, "ent": "stole", "catw": "weapon", "stmt": "Stole is a type of weapon."}, {"id": 638, "ent": "stapler", "catw": "boat", "stmt": "Stapler is a type of boat."}, {"id": 639, "ent": "ski", "catw": "device", "stmt": "Ski is a type of device."}, {"id": 640, "ent": "materiel", "catw": "equipment", "stmt": "Materiel is a type of equipment."}, {"id": 641, "ent": "rheumatic", "catw": "living thing", "stmt": "Rheumatic is a type of living thing."}, {"id": 642, "ent": "margarita", "catw": "dish", "stmt": "Margarita is a type of dish."}, {"id": 643, "ent": "dredge", "catw": "animal", "stmt": "Dredge is a type of animal."}, {"id": 644, "ent": "private", "catw": "fruit", "stmt": "Private is a type of fruit."}, {"id": 645, "ent": "chalet", "catw": "car", "stmt": "Chalet is a type of car."}, {"id": 646, "ent": "locket", "catw": "device", "stmt": "Locket is a type of device."}, {"id": 647, "ent": "earl", "catw": "animal", "stmt": "Earl is a type of animal."}, {"id": 648, "ent": "blazer", "catw": "weapon", "stmt": "Blazer is a type of weapon."}, {"id": 649, "ent": "jammer", "catw": "body part", "stmt": "Jammer is a type of body part."}, {"id": 650, "ent": "crustacean", "catw": "food", "stmt": "Crustacean is a type of food."}, {"id": 651, "ent": "android", "catw": "worker", "stmt": "Android is a type of worker."}, {"id": 652, "ent": "cowgirl", "catw": "animal", "stmt": "Cowgirl is a type of animal."}, {"id": 653, "ent": "entrepreneur", "catw": "food", "stmt": "Entrepreneur is a type of food."}, {"id": 654, "ent": "implant", "catw": "body part", "stmt": "Implant is a type of body part."}, {"id": 655, "ent": "camcorder", "catw": "device", "stmt": "Camcorder is a type of device."}, {"id": 656, "ent": "macadamia", "catw": "living thing", "stmt": "Macadamia is a type of living thing."}, {"id": 657, "ent": "waterfowl", "catw": "game", "stmt": "Waterfowl is a type of game."}, {"id": 658, "ent": "celibate", "catw": "person", "stmt": "Celibate is a type of person."}, {"id": 659, "ent": "rhinoceros", "catw": "person", "stmt": "Rhinoceros is a type of person."}, {"id": 660, "ent": "blazer", "catw": "man-made object", "stmt": "Blazer is a type of man-made object."}, {"id": 661, "ent": "retriever", "catw": "game", "stmt": "Retriever is a type of game."}, {"id": 662, "ent": "antelope", "catw": "footwear", "stmt": "Antelope is a type of footwear."}, {"id": 663, "ent": "wally", "catw": "sport", "stmt": "Wally is a type of sport."}, {"id": 664, "ent": "usurper", "catw": "game", "stmt": "Usurper is a type of game."}, {"id": 665, "ent": "cultivar", "catw": "material", "stmt": "Cultivar is a type of material."}, {"id": 666, "ent": "chalet", "catw": "game", "stmt": "Chalet is a type of game."}, {"id": 667, "ent": "kitchen", "catw": "equipment", "stmt": "Kitchen is a type of equipment."}, {"id": 668, "ent": "matchbox", "catw": "toy", "stmt": "Matchbox is a type of toy."}, {"id": 669, "ent": "homeowner", "catw": "person", "stmt": "Homeowner is a type of person."}, {"id": 670, "ent": "candlestick", "catw": "tool", "stmt": "Candlestick is a type of tool."}, {"id": 671, "ent": "priestess", "catw": "dog", "stmt": "Priestess is a type of dog."}, {"id": 672, "ent": "toddler", "catw": "mammal", "stmt": "Toddler is a type of mammal."}, {"id": 673, "ent": "detonator", "catw": "man-made object", "stmt": "Detonator is a type of man-made object."}, {"id": 674, "ent": "ax", "catw": "man-made object", "stmt": "Ax is a type of man-made object."}, {"id": 675, "ent": "lingerie", "catw": "equipment", "stmt": "Lingerie is a type of equipment."}, {"id": 676, "ent": "biochemist", "catw": "worker", "stmt": "Biochemist is a type of worker."}, {"id": 677, "ent": "clary", "catw": "weapon", "stmt": "Clary is a type of weapon."}, {"id": 678, "ent": "juvenile", "catw": "person", "stmt": "Juvenile is a type of person."}, {"id": 679, "ent": "psychologist", "catw": "worker", "stmt": "Psychologist is a type of worker."}, {"id": 680, "ent": "obstetrician", "catw": "person", "stmt": "Obstetrician is a type of person."}, {"id": 681, "ent": "biologist", "catw": "worker", "stmt": "Biologist is a type of worker."}, {"id": 682, "ent": "camcorder", "catw": "room", "stmt": "Camcorder is a type of room."}, {"id": 683, "ent": "aristocrat", "catw": "fruit", "stmt": "Aristocrat is a type of fruit."}, {"id": 684, "ent": "tripod", "catw": "equipment", "stmt": "Tripod is a type of equipment."}, {"id": 685, "ent": "ashtray", "catw": "metal", "stmt": "Ashtray is a type of metal."}, {"id": 686, "ent": "atheist", "catw": "dog", "stmt": "Atheist is a type of dog."}, {"id": 687, "ent": "airship", "catw": "body part", "stmt": "Airship is a type of body part."}, {"id": 688, "ent": "greenhouse", "catw": "container", "stmt": "Greenhouse is a type of container."}, {"id": 689, "ent": "canoe", "catw": "vehicle", "stmt": "Canoe is a type of vehicle."}, {"id": 690, "ent": "ship", "catw": "tool", "stmt": "Ship is a type of tool."}, {"id": 691, "ent": "jodhpur", "catw": "vehicle", "stmt": "Jodhpur is a type of vehicle."}, {"id": 692, "ent": "coffin", "catw": "footwear", "stmt": "Coffin is a type of footwear."}, {"id": 693, "ent": "wrestler", "catw": "athlete", "stmt": "Wrestler is a type of athlete."}, {"id": 694, "ent": "dominatrix", "catw": "vehicle", "stmt": "Dominatrix is a type of vehicle."}, {"id": 695, "ent": "caribou", "catw": "game", "stmt": "Caribou is a type of game."}, {"id": 696, "ent": "harmonica", "catw": "equipment", "stmt": "Harmonica is a type of equipment."}, {"id": 697, "ent": "genitalia", "catw": "vegetable", "stmt": "Genitalia is a type of vegetable."}, {"id": 698, "ent": "anesthesiologist", "catw": "worker", "stmt": "Anesthesiologist is a type of worker."}, {"id": 699, "ent": "golf", "catw": "equipment", "stmt": "Golf is a type of equipment."}]''')
WK6 = json.load(open(E6_WK))
for it in ITEMS: it['wk_defs'] = WK6.get(it['ent'], [])
print(len(ITEMS), '個', ' 物の語の説明文あり:', sum(1 for it in ITEMS if it['wk_defs']))

## 1. Wiktionaryのダンプを取って読む（25〜45分、GPUは使わない）

In [ ]:
wkd_path = f"{OUT}/E7_wiktionary_dump_nouns.json"
if os.path.exists(wkd_path):
    WKD = json.load(open(wkd_path)); print("ドライブから読み込み:", len(WKD))
else:
    dump = "/content/enwiktionary.xml.bz2"
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", dump, DUMP_URL], check=True)
    print("取れた大きさ:", round(os.path.getsize(dump) / 1e9, 2), "GB")
    t = time.time()
    with bz2.open(dump, "rb") as f:
        WKD = parse_dump(f)
    print("読み終わり:", len(WKD), "語", f"{time.time() - t:.0f}秒")
    json.dump(WKD, open(wkd_path, "w"), ensure_ascii=False)
print("例 cat:", WKD.get("cat", [])[:2])

## 2. 辞書をたどって網の材料を作る

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])
G = build_graph(ITEMS, lambda w, lim: WKD.get(w, [])[:lim], nlp)
steps, arrive = nli_jobs(ITEMS, G)
print("語の数:", len(G), " 段の組:", len(steps), " 着きの組:", len(arrive))
for w in ["kitten", "granddaughter", "barbell"]:
    if w in G: print(w, [x["genus"] for x in G[w]])

## 3. NLIで判定する（5〜10分、GPU）

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["neutral"], lab["contradiction"]]
def nli_probs(P, H, B=64):
    out = []
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER]
        out += pr.cpu().numpy().round(4).tolist()
        if (i // B) % 200 == 0: print(f"{i}/{len(P)}")
    return out
t = time.time()
S = nli_probs([premise(w, G[w][k]["def"]) for w, k in steps], [hyp(w, G[w][k]["genus"]) for w, k in steps])
A = nli_probs([premise(w, G[w][k]["def"]) for w, k, C in arrive], [hyp(w, C) for w, k, C in arrive])
res = {"meta": {"nli": NLI_ID, "dump": DUMP_URL, "date": time.strftime("%Y-%m-%d"), "n_items": len(ITEMS)},
       "G": G, "steps": [[w, k, p[0], p[2]] for (w, k), p in zip(steps, S)],
       "arrive": [[w, k, C, p[0], p[2]] for (w, k, C), p in zip(arrive, A)]}
path = f"{OUT}/E7_results.json"
json.dump(res, open(path, "w"), ensure_ascii=False)
print("保存:", path, round(os.path.getsize(path) / 1e6, 1), "MB", f"{time.time() - t:.0f}秒")